# Actividad 6 — Desigualdad espacial en la Región Metropolitana

**INF-497 · Análisis de Datos Espaciales**
**Sesión práctica · 29 abril (2 bloques)**

## Contexto

En la clase de hoy aplicamos a la RM las técnicas vistas en `06_desigualdad_espacial.ipynb` (basado en cap. 9 del libro *Geographic Data Science with Python*).

Trabajaremos con datos de la encuesta **CASEN 2017 y 2022** agregados a las **52 comunas** de la Región Metropolitana, ya preparados por el preámbulo. Tendremos dos definiciones de ingreso por comuna:

- **Ingreso autónomo per cápita** (`ypc_aut`): pre-transferencias estatales (sueldos, rentas, capital).
- **Ingreso total per cápita** (`ypc_tot`): post-transferencias (incluye subsidios y pensiones).

La gracia es que con esos datos podemos responder dos preguntas distintas:

1. *Temporal:* ¿la geografía de la desigualdad en RM cambió entre 2017 y 2022?
2. *Estructural:* ¿cuánto reducen las transferencias del Estado la desigualdad **espacial** (no solo la individual)?

## Reglas

- Trabajo en **parejas**, una entrega por pareja al Aula Virtual antes del fin de bloque.
- Suban este notebook con sus respuestas (código + texto en markdown) ejecutado.
- Las preguntas marcadas **🟠 Interpretación** se responden en celdas markdown — no basta con código.
- Si una comuna tiene `n < 200` en la muestra CASEN, el promedio comunal es ruidoso. Pueden filtrarlas o discutir el efecto en sus respuestas.

## Requisito previo

Ejecuten una vez el notebook **`06_actividad6_preambulo.ipynb`** para generar el archivo `datos/external/casen_rm/casen_rm_comunas.gpkg`. Después no lo necesitan más.

## 0. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns

import esda
from libpysal import weights
from inequality.gini import Gini, Gini_Spatial
from inequality.theil import Theil, TheilD

warnings.filterwarnings("ignore")
sns.set_context("notebook")

# Detectar root del proyecto
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

gdf = gpd.read_file(ROOT / "datos/external/casen_rm/casen_rm_comunas.gpkg")
print(f"{len(gdf)} comunas RM cargadas | CRS: {gdf.crs}")
gdf.head(3)

---

## Ejercicio 1 — Exploración inicial

Antes de calcular métricas, vamos a ver con qué tipo de distribución estamos trabajando y qué patrones espaciales saltan a la vista.

### 1.A — Histogramas

Hagan un grid de **2×2** con histogramas de las cuatro variables de ingreso (`ypc_aut_2017`, `ypc_aut_2022`, `ypc_tot_2017`, `ypc_tot_2022`). Usen `seaborn.histplot` con `kde=True`. Pongan títulos claros.

In [ ]:
# Tu código aquí


### 1.B — Mapas coropléticos

Hagan un grid de **2×2** con mapas coropléticos de las mismas cuatro variables. Usen `scheme="Quantiles"`, `k=5`, una paleta razonable (`viridis` o similar). Asegúrense de que los cuatro mapas usen **la misma escala de colores** para que sean comparables visualmente — pueden lograrlo definiendo `vmin`/`vmax` comunes.

> 💡 Tip: para escala común, calculen `vmin = gdf[cols].values.min()` y `vmax = gdf[cols].values.max()` antes de plotear.

In [ ]:
# Tu código aquí


### 1.C — 🟠 Interpretación

Respondan en una celda markdown:

1. ¿La distribución del ingreso por comuna es simétrica o sesgada? ¿En qué dirección? ¿Qué implica eso para usar la **media** vs la **mediana**?
2. Sin calcular nada todavía: **al ojo**, ¿dónde están las comunas más ricas y dónde las más pobres? ¿Hay un patrón espacial claro?
3. Comparando los dos años: ¿se ve algún cambio obvio en la geografía de la riqueza?

**Respuesta 1.C:** *(escriban acá)*

---

## Ejercicio 2 — Medidas globales de desigualdad

Calculen los tres índices clásicos para las cuatro combinaciones (autónomo / total × 2017 / 2022) y comparen.

### 2.A — Gini, Theil y Ratio 20:20

Construyan un DataFrame `indices` con índice las 4 combinaciones (`aut_2017`, `aut_2022`, `tot_2017`, `tot_2022`) y columnas `gini`, `theil`, `ratio_20_20`.

Usen:
- `inequality.gini.Gini(valores).g` para Gini
- `inequality.theil.Theil(valores).T` para Theil
- Cálculo manual del ratio 20:20: `Q80 / Q20`

In [ ]:
# Tu código aquí
# indices = pd.DataFrame(...)
# indices


### 2.B — Curvas de Lorenz comparadas

Grafiquen en **un solo eje** las 4 curvas de Lorenz (use distintos colores y tipos de línea — autónomo en sólido, total en punteado, 2017 en azul, 2022 en naranja, por ejemplo). Incluyan la diagonal de igualdad perfecta.

Pueden reutilizar la función `lorenz(y)` del notebook 06.

In [ ]:
# Tu código aquí


### 2.C — 🟠 Interpretación

1. ¿La desigualdad **subió o bajó** entre 2017 y 2022 según los tres índices? ¿Coinciden?
2. Comparen autónomo vs total dentro del **mismo año**. ¿Cuánto reducen las transferencias del Estado la desigualdad **entre comunas**? Cuantifíquenlo (ej. caída porcentual del Gini).
3. ¿Por qué el ratio 20:20 puede dar una historia distinta del Gini? Piensen en qué parte de la distribución mira cada uno.

**Respuesta 2.C:** *(escriban acá)*

---

## Ejercicio 3 — Autocorrelación espacial: Moran's I

Hasta acá los índices ignoran completamente la geografía. Ahora la incorporamos.

### 3.A — Matriz de pesos y Moran's I

1. Construyan una matriz de pesos **Queen** sobre `gdf`.
2. Calculen Moran's I del ingreso autónomo per cápita para 2017 y para 2022.
3. Repitan con ingreso total. Reporten `I` y `p_sim` para los 4 casos.

In [ ]:
# Tu código aquí
# wq = weights.Queen.from_dataframe(gdf)
# wq.transform = "r"


### 3.B — Diagrama de Moran

Para `ypc_aut_2017`, hagan el **gráfico de dispersión de Moran**: ingreso estandarizado en X, su rezago espacial en Y, con líneas en 0 y la recta de regresión cuya pendiente es Moran's I.

Identifiquen visualmente las comunas en cada cuadrante (HH, LL, HL, LH). Pueden marcar 2-3 comunas con `ax.annotate(nom_comuna, ...)`.

In [ ]:
# Tu código aquí


### 3.C — 🟠 Interpretación

1. ¿Hay autocorrelación espacial significativa? ¿Positiva o negativa?
2. Comparando autónomo vs total: ¿la autocorrelación cambia mucho? ¿Eso qué nos dice sobre cómo operan las transferencias en el espacio?
3. Comparando 2017 vs 2022: ¿la estructura espacial se está fortaleciendo o debilitando?

**Respuesta 3.C:** *(escriban acá)*

---

## Ejercicio 4 — Descomposición regional del Theil

A diferencia del notebook 06 (que descomponía por las 8 regiones censales de EE.UU.), nosotros descompondremos por las **6 provincias** de la RM:

| Provincia | N° comunas |
|---|---|
| Santiago | 32 |
| Cordillera | 3 |
| Chacabuco | 3 |
| Maipo | 4 |
| Melipilla | 5 |
| Talagante | 5 |

La pregunta es: ¿la desigualdad en RM viene mayoritariamente de diferencias **entre provincias** (Santiago vs el cinturón rural) o **dentro de cada provincia** (ej. comunas pobres y ricas dentro del Gran Santiago)?

### 4.A — TheilD por provincia

Calculen `TheilD(valores, gdf['nom_prov'].values)` para los 4 casos (autónomo/total × 2017/2022). Construyan un DataFrame con columnas `theil_total`, `theil_entre`, `theil_dentro`, `prop_entre = entre / total`.

In [ ]:
# Tu código aquí


### 4.B — Visualización

Hagan un gráfico de barras agrupadas mostrando para los 4 casos los componentes `theil_entre` y `theil_dentro` apilados (stacked bar), o lado a lado. El eje Y es el valor del Theil; el eje X las 4 combinaciones.

In [ ]:
# Tu código aquí


### 4.C — 🟠 Interpretación

1. ¿Qué componente domina, *entre* provincias o *dentro* de provincias? ¿Tiene sentido dado el tamaño de cada provincia?
2. ¿Cómo cambia esta proporción con las transferencias? ¿Las transferencias atacan más la desigualdad entre provincias o dentro?
3. Comparen con el caso EE.UU. del notebook 06, donde la proporción *entre regiones* era ~30 % y bajaba con el tiempo. ¿Qué pasa en RM?

**Respuesta 4.C:** *(escriban acá)*

---

## Ejercicio 5 — Gini espacial: vecinos vs no-vecinos

Última pieza: descomponer el Gini en **diferencias entre comunas vecinas** vs **diferencias entre comunas no vecinas**, sin usar la división administrativa por provincia.

### 5.A — Gini espacial 2017 vs 2022

Calculen `Gini_Spatial(valores, wq)` para autónomo 2017 y autónomo 2022. **Importante:** la matriz de pesos debe estar en transformación binaria — pongan `wq.transform = "B"` antes de llamar a `Gini_Spatial`.

Reporten para cada caso:
- `g` (Gini total)
- `wcg_share` (proporción del Gini que aportan los pares no-vecinos)
- `p_sim` (significancia)

In [ ]:
# Tu código aquí
# wq.transform = "B"


### 5.B — Diferencias entre vecinos en el tiempo

Calculen el componente normalizado de "diferencias cercanas":

```python
def diferencias_cercanas(valores, w):
    gs = Gini_Spatial(valores, w)
    denom = 2 * np.mean(valores) * w.n ** 2
    return gs.wg / denom
```

Apliquen a las 4 combinaciones y compáreelo con los Moran's I del Ejercicio 3 (un scatter o tabla).

In [ ]:
# Tu código aquí


### 5.C — 🟠 Interpretación de cierre

1. ¿La mayor parte de la desigualdad viene de diferencias entre **vecinos** o entre comunas **lejanas**? ¿Eso qué les dice sobre si la RM tiene "barrios" socioeconómicos claramente delimitados?
2. ¿Hay coherencia entre lo que dice Moran's I y el Gini espacial? Si no, ¿por qué?
3. **Pregunta integradora final**: con todo lo calculado, escriban en 5-7 líneas la "historia" de la desigualdad espacial en la RM 2017→2022. Asegúrense de mencionar:
   - cómo cambió la desigualdad agregada,
   - cómo cambió su estructura espacial,
   - qué papel juegan las transferencias del Estado.

**Respuesta 5.C:** *(escriban acá)*

---

## Entrega

Antes de subir el notebook al Aula Virtual:

- ✅ Reinicien el kernel y ejecuten todo de cero (`Kernel → Restart & Run All`) para verificar reproducibilidad.
- ✅ Confirmen que las 5 preguntas de interpretación están respondidas.
- ✅ Renombren el archivo a `06_actividad6_<apellido1>_<apellido2>.ipynb`.

## Para profundizar (no obligatorio)

- Repliquen el análisis usando `mediana_aut_2017` y `mediana_aut_2022` (que son más robustas a outliers que el promedio). ¿Cambia la conclusión?
- Filtren las comunas con `n < 200` en CASEN 2022 y rehagan los índices. ¿Cuán sensibles son los resultados al ruido muestral?
- Comparen el Gini comunal RM-2022 con el Gini personal de Chile reportado por el Ministerio de Desarrollo Social (~0.41 en 2022). ¿Por qué son tan distintos?